# 4.1 Item-Based Collaborative Filtering

This notebook implements an **Item-Based Collaborative Filtering** recommender system for the RetailRocket implicit-feedback dataset.

The model recommends items based on their similarity to items the user has previously interacted with.

# 4.2 Problem Definition

We want to recommend items that a user is likely to interact with in the near future.

The dataset contains implicit feedback such as:

- `view`
- `addtocart`
- `transaction`

These interactions are converted into weighted user-item interactions:

- `view` → 1
- `addtocart` → 3
- `transaction` → 5

Unlike the popularity-based baseline, Item-Based Collaborative Filtering uses the user's interaction history to generate personalized recommendations.

The goal is to rank unseen items that are similar to the items the user has already interacted with.

# 4.3 Item-Based Collaborative Filtering

### Core Idea

The main idea of Item-Based Collaborative Filtering is to recommend items that are similar to the items the user has already interacted with.

Instead of learning a latent representation of the user, as in ALS, we first model relationships between items.

The recommendation process is:

User history → Similar items → Aggregate scores → Top-K recommendations

### User-Item Matrix

We start with the user-item interaction matrix.

Rows represent users and columns represent items. The matrix value represents the strength of the observed interaction.

For example:

| | Item A | Item B | Item C | Item D |
|---|---:|---:|---:|---:|
| User 1 | 1 | 1 | 0 | 0 |
| User 2 | 1 | 0 | 1 | 0 |
| User 3 | 0 | 1 | 1 | 1 |
| User 4 | 1 | 0 | 0 | 1 |

A zero means that no interaction was observed. It does not represent explicit negative feedback.

### Item Representations

To compare items, we look at their interaction patterns across users.

For example:

$$
Item\ A = [1, 1, 0, 1]
$$

$$
Item\ B = [1, 0, 1, 0]
$$

Items with similar interaction patterns will have similar representations.

### Item-Item Similarity

We calculate a similarity score between pairs of items based on their user interaction patterns.

This gives us an item-item similarity matrix:

$$
S_{ij} = sim(i,j)
$$

For a large catalog, storing all pairwise similarities is expensive, so in practice we keep only the most similar items for each item.

### Cosine Similarity

We use cosine similarity to measure the similarity between item vectors:

$$
sim(x,y)=
\frac{x \cdot y}
{\|x\|\|y\|}
$$

Higher values indicate more similar interaction patterns.

### Recommendation Score

Suppose a user has interacted with items from the set:

$$
H_u
$$

For a candidate item $j$, we aggregate its similarity to the items from the user's history.

With weighted implicit feedback:

$$
score(u,j)=
\sum_{i \in H_u}
w_{ui}\cdot sim(i,j)
$$

where:

- $H_u$ is the user's interaction history
- $w_{ui}$ is the interaction weight
- $sim(i,j)$ is the similarity between items

We calculate this score for candidate items, exclude items already seen by the user, and rank the remaining items to produce the Top-K recommendations.

# 4.4 Why Item-Based CF?

Item-Based Collaborative Filtering provides an interpretable form of personalization.

Compared with the previous approaches:

- **Baseline** mainly relies on item popularity.
- **ALS** learns latent user and item representations.
- **Item-Based CF** directly models item-to-item relationships.

This makes the relationships between recommendations easier to interpret.

# 4.5 Limitations

Item-Based Collaborative Filtering has several limitations:

- computing item-item similarities can be expensive for a large catalog
- new items have little or no interaction history
- popular items may dominate similarity calculations
- recommendations depend strongly on the user's existing interaction history

# 4.6 Data Preparation

We use the same interaction processing and temporal evaluation procedure as in the previous notebooks.

The raw RetailRocket events are first converted into weighted user-item interactions.

Interaction weights are defined as:

- `view` → 1
- `addtocart` → 3
- `transaction` → 5

Multiple events between the same user and item are aggregated into a single interaction. The maximum timestamp is preserved to determine when the interaction last occurred.

The resulting interaction data is then divided into 10 consecutive one-day temporal evaluation windows.

In [1]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

events_path = DATA_DIR / "events.csv"
events = pd.read_csv(events_path)

events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [3]:
events.shape

(2756101, 5)

In [4]:
events.dtypes

timestamp          int64
visitorid          int64
event             object
itemid             int64
transactionid    float64
dtype: object

In [5]:
event_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5,
}

events["weight"] = events["event"].map(event_weights)

events[["event", "weight"]].drop_duplicates()

,event,weight
0,view,1
17,addtocart,3
130,transaction,5


### Interaction Aggregation

A user can interact with the same item multiple times.

We aggregate all events for the same user-item pair into a single interaction:

- interaction weights are summed
- the maximum timestamp is preserved

The maximum timestamp is used to determine when the user-item interaction was last observed during temporal evaluation.

In [6]:
interactions = (
    events
    .groupby(["visitorid", "itemid"])
    .agg(
        weight=("weight", "sum"),
        timestamp=("timestamp", "max")
    )
    .reset_index()
)

interactions["datetime"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms"
)

interactions.head()

,visitorid,itemid,weight,timestamp,datetime
0,0,67045,1,1442004917175,2015-09-11 20:55:17.175
1,0,285930,1,1442004589439,2015-09-11 20:49:49.439
2,0,357564,1,1442004759591,2015-09-11 20:52:39.591
3,1,72028,1,1439487966444,2015-08-13 17:46:06.444
4,2,216305,2,1438971463170,2015-08-07 18:17:43.170


### User and Item Indices

We create global integer indices for users and items.

The same indices are used across all temporal evaluation windows and recommendation models.

This allows us to use sparse matrices efficiently and keeps the evaluation setup consistent across experiments.

In [7]:
interactions["user_idx"], user_mapping = pd.factorize(
    interactions["visitorid"]
)

interactions["item_idx"], item_mapping = pd.factorize(
    interactions["itemid"]
)

In [8]:
n_users = interactions["user_idx"].nunique()
n_items = interactions["item_idx"].nunique()

n_users, n_items

(1407580, 235061)

### Temporal Evaluation

Random train/test splitting is not appropriate for this recommendation task because we want to predict future user behavior.

We use the same temporal evaluation scheme as in the previous notebooks.

The evaluation consists of **10 consecutive one-day test windows**.

For each window:

- training data contains interactions observed before the test period
- test data contains interactions from the following day
- users not observed during training are excluded from evaluation
- items not observed during training are excluded from evaluation

All model-specific computations are performed using only the corresponding training period.

In [9]:
n_windows = 10
test_days = 1

max_date = interactions["datetime"].max()

windows = []

for i in range(n_windows, 0, -1):

    test_start = max_date - pd.DateOffset(days=i)
    test_end = test_start + pd.DateOffset(days=test_days)

    train = interactions[
        interactions["datetime"] < test_start
    ].copy()

    test = interactions[
        (interactions["datetime"] >= test_start)
        & (interactions["datetime"] < test_end)
    ].copy()

    known_users = train["user_idx"].unique()
    known_items = train["item_idx"].unique()

    test = test[
        test["user_idx"].isin(known_users)
        & test["item_idx"].isin(known_items)
    ].copy()

    windows.append({
        "iteration": n_windows - i + 1,
        "train": train,
        "test": test,
        "train_end": test_start,
        "test_start": test_start,
        "test_end": test_end,
    })

### Temporal Split Validation

We verify that there is no temporal leakage between training and test periods and that all evaluated users and items are known during training.

In [10]:
for window in windows:

    train = window["train"]
    test = window["test"]

    print(
        f"Window {window['iteration']:2d} | "
        f"Train: {window['train_end'].date()} | "
        f"Test: {window['test_start'].date()} → "
        f"{window['test_end'].date()} | "
        f"Train interactions: {len(train):,} | "
        f"Test interactions: {len(test):,}"
    )

Window  1 | Train: 2015-09-08 | Test: 2015-09-08 → 2015-09-09 | Train interactions: 1,999,142 | Test interactions: 2,880
Window  2 | Train: 2015-09-09 | Test: 2015-09-09 → 2015-09-10 | Train interactions: 2,015,790 | Test interactions: 3,319
Window  3 | Train: 2015-09-10 | Test: 2015-09-10 → 2015-09-11 | Train interactions: 2,033,186 | Test interactions: 3,433
Window  4 | Train: 2015-09-11 | Test: 2015-09-11 → 2015-09-12 | Train interactions: 2,051,702 | Test interactions: 2,959
Window  5 | Train: 2015-09-12 | Test: 2015-09-12 → 2015-09-13 | Train interactions: 2,067,832 | Test interactions: 2,041
Window  6 | Train: 2015-09-13 | Test: 2015-09-13 → 2015-09-14 | Train interactions: 2,080,882 | Test interactions: 1,965
Window  7 | Train: 2015-09-14 | Test: 2015-09-14 → 2015-09-15 | Train interactions: 2,094,902 | Test interactions: 2,889
Window  8 | Train: 2015-09-15 | Test: 2015-09-15 → 2015-09-16 | Train interactions: 2,112,076 | Test interactions: 2,988
Window  9 | Train: 2015-09-16 | 

# 4.7 Item-Based CF Implementation

For each temporal training window, we build an item-user interaction matrix and use it to calculate item-to-item cosine similarity.

The model stores only the Top-N most similar items for each item.

This avoids keeping the full item-item similarity matrix and makes the recommendation step more efficient.

### Item-User Matrix

The training data is represented as a sparse item-user matrix:

- rows represent items
- columns represent users
- values represent interaction weights

The matrix is constructed separately for each temporal training window.

In [11]:
train = windows[0]["train"]

item_user_matrix = csr_matrix(
    (
        train["weight"],
        (
            train["item_idx"],
            train["user_idx"]
        )
    ),
    shape=(n_items, n_users),
    dtype=np.float64
)

item_user_matrix.shape

(235061, 1407580)

### Item Vector Normalization

Cosine similarity can be computed as the dot product of L2-normalized vectors.

For each item vector $x$:

$$
\hat{x} = \frac{x}{\|x\|}
$$

Then:

$$
sim(x,y) = \hat{x}\cdot\hat{y}
$$

This allows us to compute cosine similarity using sparse matrix multiplication.

In [12]:
from sklearn.preprocessing import normalize

item_user_normalized = normalize(
    item_user_matrix,
    norm="l2",
    axis=1
)

In [13]:
item_user_normalized.shape

(235061, 1407580)

### Item-Item Similarity

The item-item similarity matrix is computed as:

$$
S = X X^T
$$

where $X$ is the normalized item-user matrix.

Because the interaction matrix is sparse, we keep the similarity matrix sparse as well.

The diagonal represents the similarity of an item with itself, so it is removed before generating recommendations.

In [14]:
item_similarity = (
    item_user_normalized
    @ item_user_normalized.T
).tocsr()

item_similarity.setdiag(0)
item_similarity.eliminate_zeros()

In [15]:
item_similarity.shape

(235061, 235061)

In [16]:
# How many nonzero similarities were actually found?
item_similarity.nnz

56942229

### Top-N Similar Items

For recommendation, we do not need every non-zero item similarity.

For each item, we keep only its Top-N most similar items.

This reduces the amount of similarity information that needs to be stored and makes recommendation generation more efficient.

In [17]:
def keep_top_n_similarity(similarity_matrix, top_n=100):
    rows = []
    cols = []
    data = []

    for item_idx in range(similarity_matrix.shape[0]):

        start = similarity_matrix.indptr[item_idx]
        end = similarity_matrix.indptr[item_idx + 1]

        item_indices = similarity_matrix.indices[start:end]
        item_scores = similarity_matrix.data[start:end]

        # Remove self-similarity.
        mask = item_indices != item_idx

        item_indices = item_indices[mask]
        item_scores = item_scores[mask]

        if len(item_scores) == 0:
            continue

        if len(item_scores) > top_n:
            top_positions = np.argpartition(
                -item_scores,
                top_n - 1
            )[:top_n]

            item_indices = item_indices[top_positions]
            item_scores = item_scores[top_positions]

        rows.extend(
            [item_idx] * len(item_indices)
        )
        cols.extend(item_indices)
        data.extend(item_scores)

    return csr_matrix(
        (data, (rows, cols)),
        shape=similarity_matrix.shape,
        dtype=np.float64
    )

In [18]:
top_n = 10

top_n_similarity = keep_top_n_similarity(
    item_similarity,
    top_n=top_n
)

In [19]:
top_n_similarity.nnz

953652

In [20]:
item_idx = 100

start = top_n_similarity.indptr[item_idx]
end = top_n_similarity.indptr[item_idx + 1]

similar_items = top_n_similarity.indices[start:end]
similarity_scores = top_n_similarity.data[start:end]

list(
    zip(similar_items, similarity_scores)
)[:10]

[(np.int32(7900), np.float64(0.022289744089852327)),
 (np.int32(8399), np.float64(0.024006146360303085)),
 (np.int32(11089), np.float64(0.01760833564610889)),
 (np.int32(14473), np.float64(0.022563248114080423)),
 (np.int32(16186), np.float64(0.08448190755542287)),
 (np.int32(19753), np.float64(0.018163064367459996)),
 (np.int32(111692), np.float64(0.07675295564533363)),
 (np.int32(120858), np.float64(0.23448415270421974)),
 (np.int32(145548), np.float64(0.08290266722896784)),
 (np.int32(178005), np.float64(0.038376477822666816))]

### ItemBasedCF Model

The complete model combines the previous steps:

1. construct the item-user interaction matrix
2. normalize item vectors
3. calculate item-item cosine similarity
4. keep the Top-N most similar items

In [21]:
class ItemBasedCF:
    def __init__(self, n_items, top_n=100):
        self.n_items = n_items
        self.top_n = top_n

        self.item_similarity = None

    def fit(self, train_matrix):
        item_user_matrix = train_matrix.T.tocsr()

        item_user_normalized = normalize(
            item_user_matrix,
            norm="l2",
            axis=1
        )

        similarity = (
            item_user_normalized
            @ item_user_normalized.T
        ).tocsr()

        similarity.setdiag(0)
        similarity.eliminate_zeros()

        self.item_similarity = keep_top_n_similarity(
            similarity,
            top_n=self.top_n
        )

        return self

### Build the Training Matrix

Before fitting the model, we construct a sparse user-item matrix from the first temporal training window.

The matrix contains only interactions available before the corresponding test period.

In [22]:
train = windows[0]["train"]

train_matrix = csr_matrix(
    (
        train["weight"],
        (
            train["user_idx"],
            train["item_idx"]
        )
    ),
    shape=(n_users, n_items),
    dtype=np.float64
)

In [23]:
model = ItemBasedCF(
    n_items=n_items,
    top_n=100
)

model.fit(train_matrix)

In [24]:
model.item_similarity.shape

(235061, 235061)

In [25]:
model.item_similarity.nnz

5043230

### Generate Recommendations

For a given user, recommendations are generated from the items in the user's interaction history.

For each historical item, we retrieve its most similar items and add their similarity scores to the candidate scores.

The contribution of each historical item is weighted by the strength of the user's interaction with it:

$$
score(u,j)=
\sum_{i \in H_u}
w_{ui}\cdot sim(i,j)
$$

Items already observed by the user are excluded from the final recommendation list.

In [49]:
class ItemBasedCF:
    def __init__(self, n_items, top_n=100):
        self.n_items = n_items
        self.top_n = top_n

        self.item_similarity = None

    def fit(self, train_matrix):

        item_user_matrix = train_matrix.T.tocsr()

        item_user_normalized = normalize(
            item_user_matrix,
            norm="l2",
            axis=1
        )

        similarity = (
            item_user_normalized
            @ item_user_normalized.T
        ).tocsr()

        similarity.setdiag(0)
        similarity.eliminate_zeros()

        self.item_similarity = keep_top_n_similarity(
            similarity,
            top_n=self.top_n
        )

        return self

    def recommend(self, user_idx, train_matrix, k=10):

        scores = np.zeros(
            self.n_items,
            dtype=np.float64
        )

        start = train_matrix.indptr[user_idx]
        end = train_matrix.indptr[user_idx + 1]

        seen_items = train_matrix.indices[start:end]
        interaction_weights = train_matrix.data[start:end]

        for item_idx, weight in zip(
            seen_items,
            interaction_weights
        ):

            start_sim = self.item_similarity.indptr[item_idx]
            end_sim = self.item_similarity.indptr[item_idx + 1]

            similar_items = self.item_similarity.indices[
                start_sim:end_sim
            ]

            similarities = self.item_similarity.data[
                start_sim:end_sim
            ]

            scores[similar_items] += (
                weight * similarities
            )

        scores[seen_items] = -np.inf

        top_indices = np.argpartition(
            -scores,
            k - 1
        )[:k]

        top_indices = top_indices[
            np.argsort(-scores[top_indices])
        ]

        return top_indices

In [50]:
model = ItemBasedCF(
    n_items=n_items,
    top_n=100
)

model.fit(train_matrix)

In [51]:
recommendations = model.recommend(
    user_idx=10,
    train_matrix=train_matrix,
    k=10
)

recommendations

array([ 59737,  59228,  11547,   6831,   9206, 209813, 209816, 209812,
       209810, 209815])

# 4.8 Temporal Evaluation

We evaluate the Item-Based Collaborative Filtering model using the same temporal evaluation procedure as in the previous notebooks.

For each of the 10 one-day test windows:

- a training matrix is built using only past interactions
- the Item-Based CF model is fitted on the training data
- recommendations are generated for users observed in the test period
- items already seen during training are excluded
- the recommendations are compared with the user's actual interactions during the test period

Using the same evaluation setup allows us to compare Item-Based CF directly with the baseline and ALS models.

In [54]:
def precision_at_k(recommended_items, relevant_items, k):
    recommended_items = recommended_items[:k]

    if len(recommended_items) == 0:
        return 0.0

    hits = len(
        set(recommended_items)
        & set(relevant_items)
    )

    return hits / k


def recall_at_k(recommended_items, relevant_items, k):
    relevant_items = set(relevant_items)

    if len(relevant_items) == 0:
        return 0.0

    recommended_items = set(
        recommended_items[:k]
    )

    hits = len(
        recommended_items
        & relevant_items
    )

    return hits / len(relevant_items)


def ndcg_at_k(recommended_items, relevant_items, k):
    relevant_items = set(relevant_items)

    if len(relevant_items) == 0:
        return 0.0

    recommended_items = recommended_items[:k]

    dcg = 0.0

    for rank, item in enumerate(
        recommended_items,
        start=1
    ):
        if item in relevant_items:
            dcg += 1 / np.log2(rank + 1)

    ideal_length = min(
        len(relevant_items),
        k
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_length + 1)
    )

    return dcg / idcg


def hit_rate_at_k(recommended_items, relevant_items, k):
    recommended_items = set(
        recommended_items[:k]
    )

    relevant_items = set(relevant_items)

    return float(
        len(
            recommended_items
            & relevant_items
        ) > 0
    )

In [55]:
results = []

for window in windows:

    train = window["train"]
    test = window["test"]

    train_matrix = csr_matrix(
        (
            train["weight"],
            (
                train["user_idx"],
                train["item_idx"]
            )
        ),
        shape=(n_users, n_items),
        dtype=np.float64
    )

    model = ItemBasedCF(
        n_items=n_items,
        top_n=100
    )

    model.fit(train_matrix)

    test_relevant = (
        test
        .groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_rate_scores = []

    for user_idx, relevant_items in test_relevant.items():

        recommendations = model.recommend(
            user_idx=user_idx,
            train_matrix=train_matrix,
            k=10
        )

        precision_scores.append(
            precision_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

        recall_scores.append(
            recall_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

        hit_rate_scores.append(
            hit_rate_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

    results.append({
        "window": window["iteration"],
        "precision@10": np.mean(precision_scores),
        "recall@10": np.mean(recall_scores),
        "ndcg@10": np.mean(ndcg_scores),
        "hit_rate@10": np.mean(hit_rate_scores),
    })

In [56]:
results_df = pd.DataFrame(results)

results_df

,window,precision@10,recall@10,ndcg@10,hit_rate@10
0,1,0.003478,0.024749,0.016794,0.030506
1,2,0.004194,0.030120,0.016714,0.036031
2,3,0.003549,0.028355,0.019177,0.033898
3,4,0.003548,0.026027,0.016143,0.033674
4,5,0.004541,0.033670,0.019947,0.042795
5,6,0.005228,0.041461,0.025807,0.049747
6,7,0.003824,0.028392,0.018306,0.034706
7,8,0.004509,0.029474,0.019326,0.037678
8,9,0.004620,0.032584,0.021615,0.042904
9,10,0.005702,0.046536,0.026988,0.055921


In [57]:
mean_metrics = results_df[
    [
        "precision@10",
        "recall@10",
        "ndcg@10",
        "hit_rate@10",
    ]
].mean()

print("Average metrics across 10 temporal windows:")
print(f"Precision@10: {mean_metrics['precision@10']:.4%}")
print(f"Recall@10:    {mean_metrics['recall@10']:.4%}")
print(f"NDCG@10:      {mean_metrics['ndcg@10']:.4%}")
print(f"Hit Rate@10:  {mean_metrics['hit_rate@10']:.4%}")

Average metrics across 10 temporal windows:
Precision@10: 0.4319%
Recall@10:    3.2137%
NDCG@10:      2.0082%
Hit Rate@10:  3.9786%


# 4.9 Results

The Item-Based Collaborative Filtering model was evaluated across 10 consecutive one-day temporal test windows.

### Item-Based CF Results

| Metric | Score |
|---|---:|
| Precision@10 | 0.43% |
| Recall@10 | 3.21% |
| NDCG@10 | 2.01% |
| Hit Rate@10 | 3.98% |

The model shows relatively stable performance across the evaluation windows, with some improvement in later test periods.

# 4.10 Comparison with Baseline and ALS

The same temporal evaluation procedure is used for all three models, making their results directly comparable.

| Model | Precision@10 | Recall@10 | NDCG@10 | Hit Rate@10 |
|---|---:|---:|---:|---:|
| Baseline | ~0.10% | ~0.55% | ~0.36% | — |
| ALS | 0.49% | 3.09% | 2.16% | 4.16% |
| Item-Based CF | 0.43% | 3.21% | 2.01% | 3.98% |

# 4.11 Conclusions

Item-Based Collaborative Filtering provides an interpretable approach to personalized recommendation based on item-to-item similarity.

The model uses the user's interaction history and the strength of those interactions to score unseen items.

On the RetailRocket dataset, Item-Based CF performs close to ALS across the main ranking metrics and substantially better than the simple baseline.

The main limitation is that item-item similarity can become computationally expensive for large catalogs and depends heavily on existing interaction history.

The next step is to move from similarity-based collaborative filtering to **pairwise ranking with Bayesian Personalized Ranking (BPR)**, where the model will optimize the relative ordering of preferred and non-preferred items directly.